# Phase 0 candidate-constrained division oracle

Parent: `diag_019_train16_final_validation_graph_export`. This CPU-only diagnostic uses same-run candidate, pre-ILP graph, and final graph artifacts from that parent, plus the fixed 16-video validation set, the exact final graphs passed to validation scoring, and the bounded pre-ILP candidate export. It evaluates cumulative families A, A+B, and A+B+C with atomic, conflict-aware edits. Absolute train-derived scores are optimistic; decisions use paired deltas and require positive headroom on both specimens.


In [ ]:
# --- Phase 0.1 : environment ---------------------------------------------------
import importlib, os, sys, glob, subprocess
from pathlib import Path

# Polars 1.x is split into a Python package plus a compiled runtime wheel.
# Prefer the broadly compatible runtime on Kaggle CPUs.
os.environ.setdefault("POLARS_PREFER_PKG", "32")

def _find(pattern):
    return sorted(glob.glob(pattern, recursive=True))

# Support-pack repo = the dir that contains scripts/predict_unet_transformer.py
_hits = _find("/kaggle/input/**/scripts/predict_unet_transformer.py")
assert _hits, "support-pack repo not found under /kaggle/input (attach the pack dataset)."
REPO = Path(_hits[0]).parent.parent
for p in (REPO / "src", REPO / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
print("REPO =", REPO)

# Dependency resolution is deliberately disabled so pip cannot replace Kaggle's
# already-imported numpy/scipy and create a binary ABI mismatch. Because --no-deps
# is used, every runtime dependency must be named explicitly. This list is the
# dependency closure validated by the v7 offline gate and the v8 submission.
PIP_SPECS = [
    "tracksdata", "pyscipopt", "ilpy>=0.5.1",
    "zarr>=3.0.10,<4", "geff>=1.1.3.1.1", "geff-spec<1.2",
    "polars>=1.36", "polars-runtime-32>=1.36",
    "blosc2", "dask", "imagecodecs", "scikit-image>=0.24",
    "pyarrow", "rustworkx>=0.17.1", "sqlalchemy>=2", "numcodecs>=0.13,<0.16",
    "donfig>=0.8", "google-crc32c>=1.5", "bidict>=0.23.1", "psygnal>=0.14",
    "rich", "networkx>=3.2.1", "pydantic>=2.11", "pydantic-core",
    "annotated-types", "typing-extensions>=4.13", "typing-inspection",
    "markdown-it-py", "pygments", "click", "cloudpickle", "fsspec",
    "partd", "locket", "toolz", "pyyaml", "ndindex", "msgpack",
    "numexpr", "deprecated", "wrapt",
]

CRITICAL_MODULES = ("tracksdata", "geff", "geff_spec", "zarr",
                    "pyscipopt", "ilpy", "donfig", "numcodecs",
                    "polars", "blosc2", "dask", "imagecodecs",
                    "pyarrow", "rustworkx", "sqlalchemy", "skimage")

def _clear_partial_imports():
    # A failed import can leave half-initialized packages in sys.modules.
    roots = set(CRITICAL_MODULES) | {"skimage"}
    for name in list(sys.modules):
        if any(name == root or name.startswith(root + ".") for root in roots):
            sys.modules.pop(name, None)
    importlib.invalidate_caches()

def _polars_binary_ok():
    # The real test: a py3-none-any polars wheel imports fine but has NO compiled
    # binary ('Polars binary is missing!') -> constructing ANY DataFrame raises
    # `PyDataFrame is not defined`. A Series/DataFrame build catches that; a bare
    # import does not.
    try:
        import polars as _pl
        _pl.DataFrame({"_x": [1]})
        return True
    except Exception:
        return False

def _import_failures():
    failures = {}
    for name in CRITICAL_MODULES:
        try:
            importlib.import_module(name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    try:
        import zarr as _zarr
        if int(_zarr.__version__.split(".")[0]) < 3:
            failures["zarr"] = f"zarr {_zarr.__version__} is too old; need >=3"
    except Exception:
        pass
    try:
        import polars as _pl
        polars_ok = (hasattr(_pl, "Float16") and _polars_binary_ok())
        if not polars_ok:
            failures["polars"] = f"polars {_pl.__version__} unusable (old or binary missing)"
    except Exception:
        pass
    return failures

def _run_pip(command, label):
    print(label)
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
    return result.returncode == 0

failures = _import_failures()
if failures:
    print("Dependency check failed:", failures)
    wheel_dirs = []
    for path in [REPO.parent / "wheels", *map(Path, _find("/kaggle/input/**/wheels"))]:
        if path.is_dir() and path not in wheel_dirs:
            wheel_dirs.append(path)

    base = [sys.executable, "-m", "pip", "install", "--no-deps"]
    installed = False
    if wheel_dirs:
        offline = base + ["--no-index"]
        for path in wheel_dirs:
            offline += ["--find-links", str(path)]
        installed = _run_pip(offline + PIP_SPECS,
                             f"Installing from offline wheels: {wheel_dirs}")
    if not installed:
        installed = _run_pip(base + PIP_SPECS, "Offline install unavailable; trying PyPI")
    if not installed:
        raise RuntimeError("Dependency installation failed; see pip output above.")

    _clear_partial_imports()
    failures = _import_failures()
    non_polars_failures = {k: v for k, v in failures.items() if k != "polars"}
    if non_polars_failures:
        raise ImportError(
            f"Dependencies still fail after installation: {non_polars_failures}"
        )

# polars binary guard: the offline wheels can carry a binary-less polars
# (polars-*-py3-none-any.whl) that SHADOWS Kaggle's working build once installed
# -> "Polars binary is missing!" -> `PyDataFrame is not defined` on every polars
# AND tracksdata DataFrame op. Install the Python package and its compiled runtime
# as a matched pair. Prefer the attached offline wheels, then use PyPI (this audit
# runs with internet ON). No-op when polars already works.
if not _polars_binary_ok():
    print("polars binary missing -> reinstalling polars + polars-runtime-32")
    _clear_partial_imports()
    polars_specs = ["polars>=1.36", "polars-runtime-32>=1.36"]
    repaired = False
    if wheel_dirs:
        command = [sys.executable, "-m", "pip", "install", "--no-deps",
                   "--force-reinstall", "--no-index"]
        for path in wheel_dirs:
            command += ["--find-links", str(path)]
        repaired = _run_pip(command + polars_specs,
                            "Repairing polars from offline wheels")
        _clear_partial_imports()
        repaired = repaired and _polars_binary_ok()
    if not repaired:
        repaired = _run_pip(
            [sys.executable, "-m", "pip", "install", "--no-deps",
             "--force-reinstall", *polars_specs],
            "Repairing polars from PyPI",
        )
    _clear_partial_imports()
    if not repaired or not _polars_binary_ok():
        raise ImportError(
            "polars still has no compiled runtime after reinstall; "
            "check that polars and polars-runtime-32 wheels have matching versions"
        )

import tracksdata, geff, zarr  # noqa: F401
import polars as _pl_check
print("tracksdata", getattr(tracksdata, "__version__", "?"),
      "| geff", getattr(geff, "__version__", "?"),
      "| zarr", getattr(zarr, "__version__", "?"),
      "| polars", _pl_check.__version__, "(binary OK)")

In [ ]:
"""Candidate-constrained Phase-0 division oracle for the fixed train16 validator.

This module is intended to run in a Kaggle notebook with the competition data,
the tracking support pack and the same-run candidate, pre-ILP graph, and final
validation graph outputs of diag_019. It never trains a model and never uses
test labels. All decisions are made on the frozen 16-video train-derived
validation set recorded by diag_019.
"""

from __future__ import annotations

import gzip
import hashlib
import json
import math
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import polars as pl
import tracksdata as td
from geff import GeffMetadata

from biohub_tracking.io import open_dataset
from biohub_tracking.metrics import evaluate as compute_metric
from biohub_tracking.metrics import node_recall, per_sample_metrics


EXPERIMENT_ID = "diag_023_train16_same_run_final_graph_oracle_reviewed"
FINAL_GRAPH_EXPERIMENT_ID = "diag_019_train16_final_validation_graph_export"
PROTOCOL = "public_0933_train16_candidate_oracle_v1"
DIVISION_WEIGHT = 0.1
MAX_MATCH_DISTANCE_UM = 7.0
BASELINE_ABS_TOL = 1e-10
K = td.DEFAULT_ATTR_KEYS


@dataclass(frozen=True)
class Action:
    mother_gt_id: int
    adds: frozenset[tuple[int, int]]
    removes: frozenset[tuple[int, int]]


def _find_experiment_output(search_root: Path, experiment_id: str) -> Path:
    candidates: list[Path] = []
    for path in search_root.rglob("metrics.json"):
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
        except (OSError, UnicodeDecodeError, json.JSONDecodeError):
            continue
        if payload.get("experiment_id") == experiment_id:
            candidates.append(path.parent)
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected exactly one attached {experiment_id} output, found {candidates}"
        )
    return candidates[0]


def _find_train_dir(search_root: Path, required_names: Iterable[str]) -> Path:
    names = tuple(required_names)
    candidates: set[Path] = set()
    for first in search_root.rglob(f"{names[0]}.geff"):
        candidates.add(first.parent)
    valid = [
        path
        for path in candidates
        if all((path / f"{name}.geff").exists() and (path / f"{name}.zarr").exists() for name in names)
    ]
    if len(valid) != 1:
        raise RuntimeError(f"Expected one competition train directory for all validator names, found {valid}")
    return valid[0]


def _load_graph(path: Path) -> td.graph.BaseGraph:
    loaded = td.graph.IndexedRXGraph.from_geff(path)
    return loaded[0] if isinstance(loaded, tuple) else loaded


def _node_rows(graph: td.graph.BaseGraph) -> dict[int, tuple[int, float, float, float]]:
    attrs = graph.node_attrs(attr_keys=[K.NODE_ID, K.T, K.Z, K.Y, K.X])
    return {
        int(row[K.NODE_ID]): (
            int(row[K.T]),
            float(row[K.Z]),
            float(row[K.Y]),
            float(row[K.X]),
        )
        for row in attrs.iter_rows(named=True)
    }


def _edge_pairs(graph: td.graph.BaseGraph) -> set[tuple[int, int]]:
    if graph.num_edges() == 0:
        return set()
    attrs = graph.edge_attrs(attr_keys=[K.EDGE_SOURCE, K.EDGE_TARGET])
    return {
        (int(row[K.EDGE_SOURCE]), int(row[K.EDGE_TARGET]))
        for row in attrs.iter_rows(named=True)
    }


def _load_final_graph(root: Path, name: str) -> tuple[
    dict[int, tuple[int, float, float, float]], set[tuple[int, int]]
]:
    graph_path = root / "final_validation_graphs" / f"{name}.json.gz"
    summary_path = root / "final_validation_graph_summary.jsonl"
    if not graph_path.is_file() or not summary_path.is_file():
        raise RuntimeError(f"Missing diag_019 final graph artifact for {name}")
    summaries = [json.loads(line) for line in summary_path.read_text(encoding="utf-8").splitlines() if line]
    matches = [row for row in summaries if row.get("dataset") == name]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one diag_019 final graph summary for {name}, found {len(matches)}")
    summary = matches[0]
    digest = hashlib.sha256(graph_path.read_bytes()).hexdigest()
    if digest != summary.get("sha256"):
        raise RuntimeError(f"Final graph hash mismatch for {name}")
    with gzip.open(graph_path, "rt", encoding="utf-8") as handle:
        payload = json.load(handle)
    expected_stage = "post_filter_output_graph_pre_validator_scoring"
    if payload.get("dataset") != name or payload.get("stage") != expected_stage:
        raise RuntimeError(f"Final graph identity or stage mismatch for {name}")
    nodes = {
        int(node_id): (int(t), float(z), float(y), float(x))
        for node_id, t, z, y, x in payload.get("nodes", [])
    }
    edges = {(int(source), int(target)) for source, target in payload.get("edges", [])}
    if not nodes or not edges:
        raise RuntimeError(f"Final graph is empty for {name}")
    if len(nodes) != int(summary.get("nodes", -1)) or len(edges) != int(summary.get("edges", -1)):
        raise RuntimeError(f"Final graph count mismatch for {name}")
    if any(source not in nodes or target not in nodes for source, target in edges):
        raise RuntimeError(f"Final graph contains an unknown endpoint for {name}")
    return nodes, edges


def _gt_children(graph: td.graph.BaseGraph) -> dict[int, list[int]]:
    children: dict[int, list[int]] = defaultdict(list)
    for source, target in _edge_pairs(graph):
        children[source].append(target)
    for values in children.values():
        values.sort()
    return children


def _make_graph(
    nodes: dict[int, tuple[int, float, float, float]],
    edges: set[tuple[int, int]],
    scale: tuple[float, float, float],
) -> tuple[td.graph.BaseGraph, dict[int, int]]:
    graph = td.graph.InMemoryGraph()
    for key in ("z", "y", "x"):
        graph.add_node_attr_key(key, pl.Float64, -999999.0)
    original_ids = sorted(nodes)
    new_ids = graph.bulk_add_nodes(
        [
            {"t": nodes[node_id][0], "z": nodes[node_id][1], "y": nodes[node_id][2], "x": nodes[node_id][3]}
            for node_id in original_ids
        ]
    )
    id_map = {old: int(new) for old, new in zip(original_ids, new_ids)}
    if edges:
        graph.add_edge_attr_key("edge_prob", pl.Float64, 0.0)
        graph.add_edge_attr_key("edge_dist", pl.Float64, 0.0)
        scale_arr = np.asarray(scale, dtype=np.float64)
        payload = []
        for source, target in sorted(edges):
            source_pos = np.asarray(nodes[source][1:], dtype=np.float64)
            target_pos = np.asarray(nodes[target][1:], dtype=np.float64)
            payload.append(
                {
                    "source_id": id_map[source],
                    "target_id": id_map[target],
                    "edge_prob": 1.0,
                    "edge_dist": float(np.linalg.norm((source_pos - target_pos) * scale_arr)),
                }
            )
        graph.bulk_add_edges(payload)
    return graph, id_map


def _n_total(gt_path: Path) -> float:
    metadata = GeffMetadata.read(gt_path)
    value = (metadata.extra or {}).get("estimated_number_of_nodes")
    if value is None:
        raise RuntimeError(f"Missing estimated_number_of_nodes in {gt_path}")
    return float(value)


def _score(
    nodes: dict[int, tuple[int, float, float, float]],
    edges: set[tuple[int, int]],
    gt_graph: td.graph.BaseGraph,
    scale: tuple[float, float, float],
    n_total: float,
    *,
    return_matching: bool = False,
) -> dict[str, Any]:
    graph, original_to_new = _make_graph(nodes, edges, scale)
    new_to_original = {new: old for old, new in original_to_new.items()}
    result = compute_metric(graph, gt_graph, scale=scale, max_distance=MAX_MATCH_DISTANCE_UM)
    recall = node_recall(graph, gt_graph)
    derived = per_sample_metrics(result, n_total, recall)
    division_denom = result.division_tp + result.division_fp + result.division_fn
    division_jaccard = result.division_tp / division_denom if division_denom else 0.0
    output: dict[str, Any] = {
        "adjusted_edge_jaccard": float(derived["adj_edge_jaccard"]),
        "division_jaccard": float(division_jaccard),
        "score": float(derived["adj_edge_jaccard"] + DIVISION_WEIGHT * division_jaccard),
        "edge_tp": int(result.edge_tp),
        "edge_fp": int(result.edge_fp),
        "edge_fn": int(result.edge_fn),
        "division_tp": int(result.division_tp),
        "division_fp": int(result.division_fp),
        "division_fn": int(result.division_fn),
        "edge_weight": int(result.edge_tp + result.edge_fp + result.edge_fn),
    }
    if return_matching:
        attrs = graph.node_attrs(attr_keys=[K.NODE_ID, K.MATCHED_NODE_ID])
        gt_to_pred: dict[int, int] = {}
        for row in attrs.iter_rows(named=True):
            matched = row[K.MATCHED_NODE_ID]
            if matched is not None and int(matched) != -1:
                gt_to_pred[int(matched)] = new_to_original[int(row[K.NODE_ID])]
        output["gt_to_pred"] = gt_to_pred
    return output


def _aggregate(rows: list[dict[str, Any]]) -> dict[str, float | int]:
    total_weight = sum(int(row["edge_weight"]) for row in rows) or 1
    adjusted = sum(float(row["adjusted_edge_jaccard"]) * int(row["edge_weight"]) for row in rows) / total_weight
    division_tp = sum(int(row["division_tp"]) for row in rows)
    division_fp = sum(int(row["division_fp"]) for row in rows)
    division_fn = sum(int(row["division_fn"]) for row in rows)
    division_denom = division_tp + division_fp + division_fn
    division = division_tp / division_denom if division_denom else 0.0
    return {
        "adjusted_edge_jaccard": adjusted,
        "division_jaccard": division,
        "score": adjusted + DIVISION_WEIGHT * division,
        "division_tp": division_tp,
        "division_fp": division_fp,
        "division_fn": division_fn,
        "edge_weight": total_weight,
    }


def _load_relevant_candidates(
    path: Path,
    wanted: set[tuple[int, int]],
    final_nodes: dict[int, tuple[int, float, float, float]],
    source_nodes: dict[int, tuple[int, float, float, float]],
    scale: tuple[float, float, float],
) -> tuple[dict[tuple[int, int], dict[str, Any]], dict[str, int]]:
    found: dict[tuple[int, int], dict[str, Any]] = {}
    audit = {
        "total_rows": 0,
        "retained_endpoint_rows": 0,
        "pruned_endpoint_rows": 0,
        "time_mismatch_rows": 0,
        "distance_mismatch_rows": 0,
    }
    scale_array = np.asarray(scale, dtype=np.float64)
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        for line in handle:
            row = json.loads(line)
            pair = (int(row["source_id"]), int(row["target_id"]))
            audit["total_rows"] += 1
            if pair[0] not in final_nodes or pair[1] not in final_nodes:
                audit["pruned_endpoint_rows"] += 1
                continue
            if pair[0] not in source_nodes or pair[1] not in source_nodes:
                raise RuntimeError(f"Candidate source graph is missing retained endpoint {pair}")
            audit["retained_endpoint_rows"] += 1
            source = source_nodes[pair[0]]
            target = source_nodes[pair[1]]
            if (
                int(row["t_src"]) != source[0]
                or int(row["t_tgt"]) != target[0]
                or final_nodes[pair[0]][0] != source[0]
                or final_nodes[pair[1]][0] != target[0]
            ):
                audit["time_mismatch_rows"] += 1
            expected_distance = float(
                np.linalg.norm(
                    (np.asarray(source[1:], dtype=np.float64) - np.asarray(target[1:], dtype=np.float64))
                    * scale_array
                )
            )
            if not math.isclose(
                float(row["distance_um"]),
                expected_distance,
                rel_tol=1e-6,
                abs_tol=1e-5,
            ):
                audit["distance_mismatch_rows"] += 1
            if pair in wanted:
                found[pair] = row
    return found, audit


def _select_nonconflicting(actions: list[Action]) -> tuple[list[Action], int]:
    selected: list[Action] = []
    claimed_mothers: set[int] = set()
    claimed_targets: dict[int, int] = {}
    selected_adds: set[tuple[int, int]] = set()
    selected_removes: set[tuple[int, int]] = set()
    conflicts = 0
    for action in sorted(actions, key=lambda item: item.mother_gt_id):
        conflict = action.mother_gt_id in claimed_mothers
        for source, target in action.adds:
            conflict |= target in claimed_targets and claimed_targets[target] != source
        conflict |= bool(action.adds & selected_removes)
        conflict |= bool(action.removes & selected_adds)
        if conflict:
            conflicts += 1
            continue
        selected.append(action)
        claimed_mothers.add(action.mother_gt_id)
        for source, target in action.adds:
            claimed_targets[target] = source
        selected_adds.update(action.adds)
        selected_removes.update(action.removes)
    return selected, conflicts


def _family_actions(
    family: str,
    gt_divisions: list[int],
    children: dict[int, list[int]],
    gt_to_pred: dict[int, int],
    base_edges: set[tuple[int, int]],
    candidate_rows: dict[tuple[int, int], dict[str, Any]],
) -> tuple[list[Action], dict[str, int]]:
    out_adj: dict[int, set[int]] = defaultdict(set)
    in_adj: dict[int, set[int]] = defaultdict(set)
    for source, target in base_edges:
        out_adj[source].add(target)
        in_adj[target].add(source)
    actions: list[Action] = []
    stats = {"gt_divisions": len(gt_divisions), "detectable": 0, "applicable": 0, "candidate_covered": 0}
    for mother_gt in gt_divisions:
        daughters_gt = children[mother_gt]
        if len(daughters_gt) != 2:
            continue
        required_gt = [mother_gt, *daughters_gt]
        if any(node_id not in gt_to_pred for node_id in required_gt):
            continue
        stats["detectable"] += 1
        mother = gt_to_pred[mother_gt]
        daughters = {gt_to_pred[node_id] for node_id in daughters_gt}
        linked_true = daughters & out_adj[mother]
        adds: set[tuple[int, int]] = set()
        removes: set[tuple[int, int]] = set()
        if family in {"A", "AB"}:
            if len(out_adj[mother]) != 1 or len(linked_true) != 1:
                continue
            missing = next(iter(daughters - linked_true))
            if family == "A" and in_adj[missing]:
                continue
            adds.add((mother, missing))
            if family == "AB":
                removes.update((source, missing) for source in in_adj[missing] if source != mother)
        elif family == "ABC":
            missing_daughters = daughters - linked_true
            wrong_children = out_adj[mother] - daughters
            if not missing_daughters and not wrong_children:
                continue
            adds.update((mother, daughter) for daughter in missing_daughters)
            removes.update((mother, child) for child in wrong_children)
            for daughter in daughters:
                removes.update((source, daughter) for source in in_adj[daughter] if source != mother)
        else:
            raise ValueError(f"Unknown family: {family}")
        stats["applicable"] += 1
        if any(pair not in candidate_rows for pair in adds):
            continue
        stats["candidate_covered"] += 1
        actions.append(Action(mother_gt, frozenset(adds), frozenset(removes)))
    return actions, stats


def _apply_actions(base_edges: set[tuple[int, int]], actions: list[Action]) -> set[tuple[int, int]]:
    removes = set().union(*(action.removes for action in actions)) if actions else set()
    adds = set().union(*(action.adds for action in actions)) if actions else set()
    return (base_edges - removes) | adds


def _baseline_matches(parent_metrics: dict[str, Any], specimen: str, aggregate: dict[str, Any]) -> bool:
    expected = parent_metrics["specimen_metrics"][specimen]
    return math.isclose(
        float(aggregate["adjusted_edge_jaccard"]),
        float(expected["adjusted_edge_jaccard"]),
        rel_tol=0.0,
        abs_tol=BASELINE_ABS_TOL,
    ) and math.isclose(
        float(aggregate["division_jaccard"]),
        float(expected["division_jaccard"]),
        rel_tol=0.0,
        abs_tol=BASELINE_ABS_TOL,
    )


def run(search_root: Path = Path("/kaggle/input"), output_path: Path = Path("/kaggle/working/metrics.json")) -> dict[str, Any]:
    started = time.monotonic()
    parent_root = _find_experiment_output(search_root, FINAL_GRAPH_EXPERIMENT_ID)
    final_graph_root = parent_root
    parent_metrics = json.loads((parent_root / "metrics.json").read_text(encoding="utf-8"))
    required_parent_keys = {"primary_metric", "specimen_metrics"}
    if not required_parent_keys.issubset(parent_metrics):
        raise RuntimeError(f"Parent metrics schema is missing {sorted(required_parent_keys - set(parent_metrics))}")
    for specimen in ("44b6", "6bba"):
        required_specimen_keys = {"adjusted_edge_jaccard", "division_jaccard", "samples"}
        available = set(parent_metrics.get("specimen_metrics", {}).get(specimen, {}))
        if not required_specimen_keys.issubset(available):
            raise RuntimeError(
                f"Parent metrics schema for {specimen} is missing "
                f"{sorted(required_specimen_keys - available)}"
            )
    validation_names = [
        name
        for specimen in ("44b6", "6bba")
        for name in parent_metrics["specimen_metrics"][specimen]["samples"]
    ]
    train_dir = _find_train_dir(search_root, validation_names)
    prediction_dir = parent_root / "tracking_repo" / "predictions" / "unknown" / "unet_transformer_val" / "split_0"
    candidate_dir = parent_root / "preilp_edge_audit"
    if not prediction_dir.is_dir() or not candidate_dir.is_dir():
        raise RuntimeError("diag_019 same-run prediction or candidate export directory is missing")

    per_video: dict[str, dict[str, Any]] = {}
    for name in validation_names:
        dataset = open_dataset(train_dir / name, normalize=False, require_tracks=True, load_image=False)
        gt = dataset.tracks
        if gt is None:
            raise RuntimeError(f"Ground truth was not loaded for {name}")
        nodes, base_edges = _load_final_graph(final_graph_root, name)
        source_nodes = _node_rows(_load_graph(prediction_dir / f"{name}.geff"))
        baseline = _score(nodes, base_edges, gt, dataset.scale, _n_total(train_dir / f"{name}.geff"), return_matching=True)
        children = _gt_children(gt)
        gt_divisions = sorted(node_id for node_id, values in children.items() if len(values) == 2)
        wanted = {
            (baseline["gt_to_pred"][mother], baseline["gt_to_pred"][daughter])
            for mother in gt_divisions
            for daughter in children[mother]
            if mother in baseline["gt_to_pred"] and daughter in baseline["gt_to_pred"]
        }
        candidate_path = candidate_dir / f"{name}.jsonl.gz"
        if not candidate_path.is_file():
            raise RuntimeError(f"Missing candidate export for {name}: {candidate_path}")
        candidates, candidate_audit = _load_relevant_candidates(
            candidate_path,
            wanted,
            nodes,
            source_nodes,
            tuple(float(value) for value in dataset.scale),
        )
        if candidate_audit["total_rows"] == 0:
            raise RuntimeError(f"Candidate export is empty for {name}")
        if candidate_audit["retained_endpoint_rows"] == 0:
            raise RuntimeError(
                f"Candidate/prediction node-id spaces have no retained overlap for {name}"
            )
        if candidate_audit["time_mismatch_rows"] or candidate_audit["distance_mismatch_rows"]:
            raise RuntimeError(
                f"Candidate/prediction node-id mapping failed metadata checks for {name}: "
                f"time_mismatches={candidate_audit['time_mismatch_rows']}, "
                f"distance_mismatches={candidate_audit['distance_mismatch_rows']}"
            )
        per_video[name] = {
            "specimen": name.split("_")[0],
            "nodes": nodes,
            "base_edges": base_edges,
            "gt": gt,
            "scale": tuple(float(value) for value in dataset.scale),
            "n_total": _n_total(train_dir / f"{name}.geff"),
            "baseline": baseline,
            "children": children,
            "gt_divisions": gt_divisions,
            "candidates": candidates,
            "candidate_audit": candidate_audit,
            "relevant_candidate_count": len(candidates),
        }

    total_relevant_candidates = sum(video["relevant_candidate_count"] for video in per_video.values())
    if total_relevant_candidates == 0:
        raise RuntimeError(
            "No GT-relevant candidate pair was found despite valid candidate endpoint ids; "
            "refusing to interpret this as zero oracle headroom"
        )

    baseline_by_specimen = {
        specimen: _aggregate([v["baseline"] for v in per_video.values() if v["specimen"] == specimen])
        for specimen in ("44b6", "6bba")
    }
    baseline_reproduced = all(
        _baseline_matches(parent_metrics, specimen, baseline_by_specimen[specimen])
        for specimen in baseline_by_specimen
    )

    family_results: dict[str, Any] = {}
    for family in ("A", "AB", "ABC"):
        scored_rows: dict[str, dict[str, Any]] = {}
        video_stats: dict[str, dict[str, int]] = {}
        for name, video in per_video.items():
            actions, stats = _family_actions(
                family,
                video["gt_divisions"],
                video["children"],
                video["baseline"]["gt_to_pred"],
                video["base_edges"],
                video["candidates"],
            )
            selected, conflicts = _select_nonconflicting(actions)
            edited_edges = _apply_actions(video["base_edges"], selected)
            score = _score(video["nodes"], edited_edges, video["gt"], video["scale"], video["n_total"])
            stats.update(
                {
                    "selected_actions": len(selected),
                    "conflicts_rejected": conflicts,
                    "adds": sum(len(action.adds) for action in selected),
                    "removes": sum(len(action.removes) for action in selected),
                }
            )
            scored_rows[name] = score
            video_stats[name] = stats
        specimen_payload: dict[str, Any] = {}
        for specimen in ("44b6", "6bba"):
            names = [name for name, video in per_video.items() if video["specimen"] == specimen]
            oracle = _aggregate([scored_rows[name] for name in names])
            baseline = baseline_by_specimen[specimen]
            counts = {
                key: sum(video_stats[name][key] for name in names)
                for key in (
                    "gt_divisions",
                    "detectable",
                    "applicable",
                    "candidate_covered",
                    "selected_actions",
                    "conflicts_rejected",
                    "adds",
                    "removes",
                )
            }
            specimen_payload[specimen] = {
                "baseline": baseline,
                "oracle": oracle,
                "delta_adjusted_edge_jaccard": oracle["adjusted_edge_jaccard"] - baseline["adjusted_edge_jaccard"],
                "delta_division_jaccard": oracle["division_jaccard"] - baseline["division_jaccard"],
                "delta_score": oracle["score"] - baseline["score"],
                **counts,
            }
        family_results[family] = {"specimens": specimen_payload, "videos": video_stats}

    abc_positive_both = all(
        family_results["ABC"]["specimens"][specimen]["delta_score"] > 0.0
        for specimen in ("44b6", "6bba")
    )
    a_positive_both = all(
        family_results["A"]["specimens"][specimen]["delta_score"] > 0.0
        for specimen in ("44b6", "6bba")
    )
    gate_passed = baseline_reproduced and abc_positive_both
    primary = min(
        family_results["ABC"]["specimens"][specimen]["delta_score"]
        for specimen in ("44b6", "6bba")
    )
    payload = {
        "schema_version": 1,
        "experiment_id": EXPERIMENT_ID,
        "primary_metric": primary,
        "baseline_primary_metric": float(parent_metrics["primary_metric"]),
        "runtime_seconds": 0.0,
        "reproducible": False,
        "methodology_valid": True,
        "validation": {
            "protocol": PROTOCOL,
            "seed": "public_0933_train16_v1",
            "warning": "Train-derived proxy with frozen feature extractors; use paired deltas and cross-specimen consistency only.",
        },
        "specimen_metrics": {
            specimen: {
                "primary_metric": family_results["ABC"]["specimens"][specimen]["delta_score"],
                "baseline_primary_metric": 0.0,
            }
            for specimen in ("44b6", "6bba")
        },
        "metrics": {
            "oracle_gate_passed": gate_passed,
            "baseline_reproduced": baseline_reproduced,
            "family_a_positive_both_specimens": a_positive_both,
            "family_abc_positive_both_specimens": abc_positive_both,
            "baseline_by_specimen": baseline_by_specimen,
            "families": family_results,
            "candidate_constraint": "diag_019 same-run top-4-per-source-or-target union within 12 um",
            "graph_source": "diag_019 post_filter_output_graph_pre_validator_scoring",
            "candidate_metadata_source": "diag_019 same-run pre-ILP prediction coordinates",
            "candidate_id_space_valid": True,
            "candidate_rows_by_video": {
                name: {
                    **video["candidate_audit"],
                    "gt_relevant": video["relevant_candidate_count"],
                }
                for name, video in per_video.items()
            },
            "total_gt_relevant_candidates": total_relevant_candidates,
            "conflict_policy": "deterministic GT-action selection with unique mother and target claims",
            "wall_clock_seconds": time.monotonic() - started,
        },
    }
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    return payload


In [ ]:
# Execute the embedded frozen analysis.
import json
from pathlib import Path

payload = run()
print(json.dumps({
    'gate': payload['metrics']['oracle_gate_passed'],
    'baseline_reproduced': payload['metrics']['baseline_reproduced'],
    'candidate_rows_by_video': payload['metrics']['candidate_rows_by_video'],
    'families': payload['metrics']['families'],
}, indent=2, sort_keys=True))


In [ ]:
# Controller output contract: the preceding analysis must write this final artifact.
metrics_path = Path('/kaggle/working/metrics.json')
assert metrics_path.is_file(), 'Oracle did not write metrics.json'
saved_metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
assert saved_metrics['experiment_id'] == 'diag_023_train16_same_run_final_graph_oracle_reviewed'
assert isinstance(saved_metrics['metrics']['oracle_gate_passed'], bool)
print('validated', metrics_path)
